# MTUS1 transcript expression in TCGA/Toil data

## Overview

This notebook retrieves transcript-level *MTUS1* expression from UCSC Xena/Toil resources. It is intended as a supplementary transcript-level check to support interpretation of gene-level *MTUS1* analyses in breast cancer cohorts.


In [ ]:
# -*- coding: utf-8 -*-
"""
Objective
1) List transcript-level (isoform) datasets on the Toil hub (UCSC Xena)
2) Extract MTUS1 (all its ENST) (i) from the probeMap, then (ii) retrieve expression
   at transcript level in an isoform-level TPM dataset.

Prerequisites:
    pip install pandas requests

Remarques importantes:
- The Xena hub is a server. Les queries passent par POST {hub}/data/
- Exact Xena query names may vary depending on the hub version.
  The code below tries several names ("datasetList", "datasets", etc.) for robustness.
- If filtering is needed later BRCA/LUAD, join with un dataset de phenotype (cohort / oncotree / etc.).
"""

from __future__ import annotations
import json
import re
from typing import Any, Dict, List, Optional, Sequence, Tuple

import pandas as pd
import requests

TOIL_HUB = "https://toil.xenahubs.net"


# ----------------------------
# Helpers: Xena API
# ----------------------------
class XenaError(RuntimeError):
    pass


def _post_json(url: str, payload: Dict[str, Any], timeout: int = 120) -> Any:
    r = requests.post(url, json=payload, timeout=timeout)
    if not r.ok:
        raise XenaError(f"HTTP {r.status_code} on {url}\nPayload={payload}\nBody={r.text[:500]}")
    # Xena often returns plain JSON (list / dict).
    try:
        return r.json()
    except Exception:
        # Xena may occasionally return JSON as text.
        try:
            return json.loads(r.text)
        except Exception as e:
            raise XenaError(f"Cannot decode JSON from {url}: {e}\nFirst 500 chars:\n{r.text[:500]}")


def xena_query(hub: str, query_name: str, args: Sequence[Any] = (), timeout: int = 120) -> Any:
    """
    Standard Xena query.
    Many hubs respond to POST {hub}/data/ with a JSON payload {"query": ..., "args": [...]}
    """
    return _post_json(f"{hub.rstrip('/')}/data/", {"query": query_name, "args": list(args)}, timeout=timeout)


def xena_query_try_many(
    hub: str,
    candidates: Sequence[Tuple[str, Sequence[Any]]],
    timeout: int = 120,
) -> Tuple[str, Any]:
    """
    Try several (query_name, args) until a usable response is obtained.
    Return (query_name_used, result).
    """
    last_err: Optional[Exception] = None
    for q, a in candidates:
        try:
            out = xena_query(hub, q, a, timeout=timeout)
            return q, out
        except Exception as e:
            last_err = e
    raise XenaError(f"None of the candidate queries worked. Last error:\n{last_err}")


# ----------------------------
# 1) List transcript-level datasets
# ----------------------------
def list_datasets(hub: str = TOIL_HUB) -> List[str]:
    """
    Return the list of dataset names on the hub.
    """
    q, out = xena_query_try_many(
        hub,
        candidates=[
            ("datasetList", ()),
            ("datasets", ()),
            ("allDatasets", ()),
        ],
    )
    # Usually a list of strings.
    if isinstance(out, list) and all(isinstance(x, str) for x in out):
        return out
    # Sometimes a dictionary or another structure.
    if isinstance(out, dict):
        # Heuristic fallback.
        for k in ("datasets", "result", "data"):
            if k in out and isinstance(out[k], list):
                return out[k]
    raise XenaError(f"Unexpected format for {q}: {type(out)}")


def dataset_info(hub: str, dataset: str) -> Dict[str, Any]:
    """
    Retrieve metadata sur un dataset (if available).
    """
    q, out = xena_query_try_many(
        hub,
        candidates=[
            ("datasetInfo", (dataset,)),
            ("datasetMetadata", (dataset,)),
            ("dataset", (dataset,)),
        ],
    )
    # Often: dict
    if isinstance(out, dict):
        return out
    # Sometimes a list containing one dictionary.
    if isinstance(out, list) and len(out) == 1 and isinstance(out[0], dict):
        return out[0]
    # fallback
    return {"_raw": out, "_query_used": q}


def list_transcript_level_datasets(hub: str = TOIL_HUB) -> pd.DataFrame:
    """
    Heuristique "transcript-level":
    - nom contient isoform / transcript / rsem_isoform
    """
    ds = list_datasets(hub)
    pat = re.compile(r"(isoform|transcript|rsem_isoform)", re.IGNORECASE)
    iso = [d for d in ds if pat.search(d)]

    rows = []
    for d in iso:
        info = dataset_info(hub, d)
        rows.append(
            {
                "hub": hub,
                "dataset": d,
                "label": info.get("label") or info.get("Label"),
                "longTitle": info.get("longTitle") or info.get("LongTitle"),
                "unit": info.get("unit") or info.get("Unit"),
                "dataSubtype": info.get("dataSubtype") or info.get("DataSubtype"),
                "probeMap": info.get("probeMap") or info.get("ProbeMap"),
            }
        )
    df = pd.DataFrame(rows).sort_values("dataset").reset_index(drop=True)
    return df


In [ ]:
# 1) List transcript-level datasets on Toil.
df_iso = list_transcript_level_datasets(TOIL_HUB)
print(df_iso[["dataset", "unit", "dataSubtype", "probeMap", "longTitle"]].head(30))

# Choose the isoform TPM dataset; this one often exists on Toil:
ISOFORM_DATASET = "TcgaTargetGtex_rsem_isoform_tpm"

In [ ]:

    

    # 2a) Find / download an isoform probeMap, then extract MTUS1 ENST identifiers.
    #     Use the probeMap listed in df_iso for the selected dataset when available.
    row = df_iso.loc[df_iso["dataset"] == ISOFORM_DATASET]
    if row.empty:
        raise XenaError(
            f"Dataset {ISOFORM_DATASET} not found in the filtered list. "
            f"Check df_iso['dataset'].unique() and choose the correct name."
        )

    probe_map_name = row["probeMap"].iloc[0]
    if not isinstance(probe_map_name, str) or probe_map_name.strip() == "" or probe_map_name == "nan":
        raise XenaError(
            "The probeMap field was not found in the metadata. "
            "Fallback: look for a dataset containing 'probeMap' in df_iso or list_datasets()."
        )

    # Download the probeMap through the hub (not through S3)
    probemap_file = download_xena_dataset(TOIL_HUB, probe_map_name, out_path="isoform_probeMap.tsv")
    mtus1_ensts = get_mtus1_ensts_from_probemap(probemap_file)
    print(f"MTUS1 transcripts found (n={len(mtus1_ensts)}):")
    print(mtus1_ensts[:30], "..." if len(mtus1_ensts) > 30 else "")

    # 2b) Retrieve TPM expression for all MTUS1 ENST identifiers.
    # Note: if all samples are retrieved TCGA+TARGET+GTEx, this can be large.
    # For testing, limit the query to the first 2,000 samples.
    all_samples = dataset_samples(TOIL_HUB, ISOFORM_DATASET)
    samples_subset = all_samples[:2000]

    expr_mtus1 = probe_values(TOIL_HUB, ISOFORM_DATASET, probes=mtus1_ensts, samples=samples_subset)
    print(expr_mtus1.shape)
    expr_mtus1.to_csv("MTUS1_isoforms_TPM_subset.csv")

    # Option: log2(TPM+1)
    expr_log = (expr_mtus1.astype(float) + 1.0).applymap(lambda x: __import__("math").log2(x))
    expr_log.to_csv("MTUS1_isoforms_log2TPMplus1_subset.csv")

